In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q --force-reinstall --no-deps git+https://github.com/Nguyenchitrai62/OpenOBB.git

import openobb
print('openobb', openobb.__version__)

In [ ]:
# 1) Pretrain backbone OpenOBB6 bang chung cat DINOv2 tren anh chua gan nhan (chi chay 1 lan, luu vao Drive)
#    COCO train2017 (118k anh, ~18 GB): da dang -> backbone tong quat. Them anh ban ve neu co.
!mkdir -p /content/coco && cd /content/coco && wget -q http://images.cocodataset.org/zips/train2017.zip && unzip -q train2017.zip && rm train2017.zip

In [ ]:
!openobb pretrain model=openobb6x data=/content/coco/train2017 epochs=20 imgsz=640 batch=64 workers=8 \
    project=/content/drive/MyDrive/Train/OpenOBB6 name=pretrain

In [ ]:
!rm -rf /content/dataset_local && unzip -q -o "/content/drive/MyDrive/Dataset/tu_ph/Wall_Color/data/dataset_wall_color_v3.zip" -d /content/dataset_local

In [ ]:
import glob, yaml; y = min(glob.glob("/content/dataset_local/**/data.yaml", recursive=True), key=len)
yaml.safe_dump({**yaml.safe_load(open(y)), "path": y.rsplit("/", 1)[0]}, open("/content/data.yaml", "w"))

In [ ]:
# 2) Train detection tu backbone da pretrain
!openobb train model=openobb6x weights=/content/drive/MyDrive/Train/OpenOBB6/pretrain/weights/last.pt \
    data=/content/data.yaml epochs=150 imgsz=1280 batch=16 project=/content/drive/MyDrive/Train/OpenOBB6 name=wall_color

In [ ]:
# 3) (tuy chon) ablation: tat mot manh, cung dieu kien. Xem docs/OPENOBB6.md muc 5
# !openobb train model=openobb6x weights=<pretrain.pt> data=/content/data.yaml epochs=150 imgsz=1280 batch=16 \
#     v6_off=cls_ctx project=/content/drive/MyDrive/Train/OpenOBB6 name=no_cls_ctx